<a href="https://colab.research.google.com/github/samyuktha07-prog/AI-Courtroom/blob/main/AI_Courtroom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip -q install gradio pandas numpy matplotlib

import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from collections import Counter

# ============================================================
# KNOWLEDGE BASE
# ============================================================

POSITIVE_WORDS = {
    "honest","truth","true","innocent","help","helped","safe",
    "proof","evidence","clear","confirmed","support","witness",
    "verified","innocence","correct","valid","reliable"
}

NEGATIVE_WORDS = {
    "lie","lied","false","fake","stolen","steal","crime","criminal",
    "guilty","threat","threatened","attack","fraud","scam","deception",
    "illegal","suspicious","missing","secret","destroyed","denied"
}

EVIDENCE_WORDS = {
    "camera","video","photo","message","email","record","receipt",
    "document","fingerprint","location","timestamp","log","witness",
    "proof","evidence","audio","call","statement"
}

STRONG_WORDS = {
    "definitely","certainly","clearly","confirmed","verified",
    "undeniable","proven","always","never"
}

UNCERTAIN_WORDS = {
    "maybe","perhaps","probably","possibly","think","guess",
    "might","could","seems","apparently","believe"
}



def tokenize(text):
    return re.findall(r"\b[a-zA-Z']+\b", text.lower())

def split_sentences(text):
    return [
        s.strip()
        for s in re.split(r"(?<=[.!?])\s+", text.strip())
        if s.strip()
    ]


def detect_claims(text):
    sents = split_sentences(text)

    claims = []

    for s in sents:

        lower = s.lower()

        if any(
            word in lower
            for word in [
                "because",
                "therefore",
                "proof",
                "evidence",
                "saw",
                "was",
                "were",
                "did",
                "didn't",
                "said",
                "told",
                "found",
                "recorded"
            ]
        ):
            claims.append(s)

    return claims[:12]



def analyze_evidence(text):

    words = tokenize(text)

    evidence_found = []

    for word in EVIDENCE_WORDS:
        if word in words:
            evidence_found.append(word)

    evidence_score = min(
        100,
        len(set(evidence_found)) * 12
    )

    return evidence_found, evidence_score



def analyze_language(text):

    words = tokenize(text)

    positive = sum(
        1 for word in words
        if word in POSITIVE_WORDS
    )

    negative = sum(
        1 for word in words
        if word in NEGATIVE_WORDS
    )

    strong = sum(
        1 for word in words
        if word in STRONG_WORDS
    )

    uncertain = sum(
        1 for word in words
        if word in UNCERTAIN_WORDS
    )

    return positive, negative, strong, uncertain



def analyze_case(case_text, prosecution, defense):

    if not case_text.strip():
        return (
            "⚠️ Enter a case first.",
            "",
            None,
            None,
            "",
            ""
        )

    combined = case_text + " " + prosecution + " " + defense

    words = tokenize(combined)
    sentences = split_sentences(combined)

    word_count = len(words)
    sentence_count = len(sentences)



    evidence, evidence_score = analyze_evidence(combined)


    p_pos, p_neg, p_strong, p_uncertain = analyze_language(
        prosecution
    )

    prosecution_strength = min(
        100,
        40
        + p_neg * 8
        + p_strong * 7
        + evidence_score * 0.25
        - p_uncertain * 5
    )



    d_pos, d_neg, d_strong, d_uncertain = analyze_language(
        defense
    )

    defense_strength = min(
        100,
        40
        + d_pos * 8
        + d_strong * 7
        + evidence_score * 0.25
        - d_uncertain * 5
    )

    prosecution_strength = max(
        0,
        round(prosecution_strength, 1)
    )

    defense_strength = max(
        0,
        round(defense_strength, 1)
    )



    balance = defense_strength - prosecution_strength

    if balance >= 15:
        verdict = "🟢 NOT GUILTY"
        verdict_reason = (
            "The defense presents stronger supporting indicators."
        )

    elif balance <= -15:
        verdict = "🔴 GUILTY"
        verdict_reason = (
            "The prosecution presents stronger indicators."
        )

    else:
        verdict = "🟡 INCONCLUSIVE"
        verdict_reason = (
            "The available information is too balanced to reach "
            "a confident conclusion."
        )

    confidence = round(
        min(
            98,
            50
            + abs(balance) * 1.5
            + evidence_score * 0.15
        ),
        1
    )



    lower = combined.lower()

    if any(
        x in lower
        for x in ["stolen", "theft", "steal", "missing"]
    ):
        case_type = "Theft / Property Case"

    elif any(
        x in lower
        for x in ["fraud", "money", "payment", "scam"]
    ):
        case_type = "Fraud / Financial Case"

    elif any(
        x in lower
        for x in ["attack", "fight", "threat"]
    ):
        case_type = "Conflict / Threat Case"

    elif any(
        x in lower
        for x in ["email", "message", "account", "password"]
    ):
        case_type = "Digital Evidence Case"

    else:
        case_type = "General Civil / Criminal Case"


    claims = detect_claims(combined)



    summary = f"""
# ⚖️ AI COURTROOM VERDICT

## {verdict}

### Case Type
**{case_type}**

### Judicial Confidence
**{confidence}%**

### Evidence Strength
**{evidence_score}/100**

### Prosecution Strength
**{prosecution_strength}/100**

### Defense Strength
**{defense_strength}/100**

---

### 👨‍⚖️ Judge's Reasoning

{verdict_reason}

The AI Courtroom compared the language patterns, evidence-related
terms, certainty indicators and arguments presented by both sides.

> ⚠️ This is an educational simulation, not real legal advice or
> an actual determination of guilt or innocence.
"""



    evidence_text = (
        ", ".join(evidence)
        if evidence
        else "No obvious evidence terms detected"
    )

    profile = f"""
## 🧾 CASE INTELLIGENCE

| Metric | Result |
|---|---:|
| Total Words | {word_count} |
| Total Statements | {sentence_count} |
| Evidence Score | {evidence_score}/100 |
| Prosecution | {prosecution_strength}/100 |
| Defense | {defense_strength}/100 |
| Confidence | {confidence}% |

### 🔎 Evidence Detected

**{evidence_text}**

### 📌 Detected Claims

"""

    if claims:
        for i, claim in enumerate(claims, 1):
            profile += f"{i}. {claim}\n\n"
    else:
        profile += "No strong factual claims detected."

    # ========================================================
    # VISUAL CHART
    # ========================================================

    chart_path = create_strength_chart(
        prosecution_strength,
        defense_strength,
        evidence_score
    )

    timeline_path = create_case_timeline(
        case_text,
        prosecution,
        defense
    )

    # ========================================================
    # FULL REPORT
    # ========================================================

    report = f"""
============================================================
                    ⚖️ AI COURTROOM
                 CASE ANALYSIS REPORT
============================================================

CASE TYPE
---------
{case_type}

FINAL VERDICT
------------
{verdict}

JUDICIAL CONFIDENCE
-------------------
{confidence}%

------------------------------------------------------------
ARGUMENT STRENGTH
------------------------------------------------------------

Prosecution Strength : {prosecution_strength}/100
Defense Strength     : {defense_strength}/100
Evidence Strength    : {evidence_score}/100

------------------------------------------------------------
EVIDENCE
------------------------------------------------------------

{evidence_text}

------------------------------------------------------------
DETECTED CLAIMS
------------------------------------------------------------

"""

    for i, claim in enumerate(claims, 1):
        report += f"{i}. {claim}\n"

    report += f"""

------------------------------------------------------------
JUDGE'S REASONING
------------------------------------------------------------

{verdict_reason}

------------------------------------------------------------
CASE STATISTICS
------------------------------------------------------------

Words       : {word_count}
Statements  : {sentence_count}

============================================================
NOTE:
This project is an educational AI simulation.
It does not provide legal advice or determine real-world
legal responsibility.

============================================================
"""

    return (
        summary,
        profile,
        chart_path,
        timeline_path,
        report,
        verdict
    )


def create_strength_chart(
    prosecution,
    defense,
    evidence
):

    labels = [
        "Prosecution",
        "Defense",
        "Evidence"
    ]

    values = [
        prosecution,
        defense,
        evidence
    ]

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.bar(labels, values)

    ax.set_ylim(0, 100)

    ax.set_ylabel("Strength / Score")

    ax.set_title(
        "⚖️ AI COURTROOM — CASE STRENGTH",
        fontsize=16
    )

    for i, value in enumerate(values):
        ax.text(
            i,
            value + 2,
            f"{value:.0f}",
            ha="center",
            fontweight="bold"
        )

    plt.tight_layout()

    path = "/content/courtroom_strength.png"

    plt.savefig(
        path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    return path



def create_case_timeline(
    case_text,
    prosecution,
    defense
):

    case_sentences = split_sentences(case_text)
    prosecution_sentences = split_sentences(prosecution)
    defense_sentences = split_sentences(defense)

    events = []

    for s in case_sentences[:4]:
        events.append(("CASE", s))

    for s in prosecution_sentences[:3]:
        events.append(("PROSECUTION", s))

    for s in defense_sentences[:3]:
        events.append(("DEFENSE", s))

    fig, ax = plt.subplots(figsize=(12, 6))

    if not events:
        events = [("CASE", "No information provided")]

    y_positions = list(
        range(len(events))
    )[::-1]

    ax.axvline(
        0,
        linewidth=3
    )

    for y, (label, sentence) in zip(
        y_positions,
        events
    ):

        ax.scatter(
            0,
            y,
            s=150
        )

        short_sentence = sentence

        if len(short_sentence) > 80:
            short_sentence = short_sentence[:77] + "..."

        ax.text(
            0.15,
            y,
            f"{label}: {short_sentence}",
            va="center",
            fontsize=10
        )

    ax.set_xlim(-0.5, 1)
    ax.set_ylim(-1, len(events))
    ax.axis("off")

    ax.set_title(
        "📜 Case Timeline",
        fontsize=17
    )

    plt.tight_layout()

    path = "/content/case_timeline.png"

    plt.savefig(
        path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.close()

    return path



sample_case_1 = """A laptop disappeared from the college laboratory
during the afternoon. The security camera recorded a student
entering the room at 3 PM. The laptop was later found in another
room. The student says they only entered the lab to collect a book."""

sample_prosecution_1 = """The camera clearly confirms that the student
entered the laboratory when the laptop disappeared. The student
was the only person recorded entering the room during that time."""

sample_defense_1 = """Entering the laboratory does not prove that the
student took the laptop. There is no direct evidence showing the
student carrying the laptop. The student also had a valid reason
for entering the room."""

sample_case_2 = """A payment was made using a disputed account.
The transaction record shows the payment occurred at 8 PM.
The accused says they were not using the account at that time."""

sample_prosecution_2 = """The transaction record confirms the payment.
The account was registered to the accused and the payment occurred
using the registered account."""

sample_defense_2 = """Account ownership does not prove who physically
made the transaction. The accused says the account credentials
could have been accessed by another person."""

# ============================================================
# LOAD SAMPLE
# ============================================================

def load_sample_1():
    return (
        sample_case_1,
        sample_prosecution_1,
        sample_defense_1
    )

def load_sample_2():
    return (
        sample_case_2,
        sample_prosecution_2,
        sample_defense_2
    )

# ============================================================
# CLEAR
# ============================================================

def clear_all():
    return (
        "",
        "",
        "",
        "",
        None,
        None,
        "",
        ""
    )


css = """
.gradio-container {
    max-width: 1250px !important;
}

.title {
    text-align: center;
    font-size: 44px;
    font-weight: 800;
}

.subtitle {
    text-align: center;
    font-size: 18px;
    opacity: 0.8;
    margin-bottom: 25px;
}

textarea {
    font-size: 15px !important;
}

button {
    font-weight: 700 !important;
}
"""



with gr.Blocks(
    title="AI Courtroom",
    css=css,
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        """
        <div class="title">⚖️ AI COURTROOM</div>

        <div class="subtitle">
        Intelligent Case Analysis & Verdict Simulation
        </div>
        """
    )

    gr.Markdown(
        """
        ### 🧑‍⚖️ Three Perspectives. One Verdict.

        **Case Facts → Evidence → Prosecution → Defense → Judge**

        This project demonstrates how natural-language analysis can
        be used to compare arguments and generate an educational
        courtroom simulation.
        """
    )


    with gr.Row():

        with gr.Column():

            case_input = gr.Textbox(
                label="📁 Case Description",
                placeholder="Describe the case...",
                lines=9
            )

            prosecution_input = gr.Textbox(
                label="🔴 Prosecution Argument",
                placeholder="Enter the prosecution's argument...",
                lines=7
            )

        with gr.Column():

            defense_input = gr.Textbox(
                label="🔵 Defense Argument",
                placeholder="Enter the defense's argument...",
                lines=9
            )

            gr.Markdown(
                """
                ### 🧑‍⚖️ Court Rules

                1. Analyze the case.
                2. Compare both arguments.
                3. Detect evidence indicators.
                4. Calculate argument strength.
                5. Generate a simulated verdict.
                """
            )


    with gr.Row():

        analyze_btn = gr.Button(
            "⚖️ RUN COURTROOM",
            variant="primary"
        )

        sample1_btn = gr.Button(
            "📱 Sample Case 1"
        )

        sample2_btn = gr.Button(
            "💳 Sample Case 2"
        )

        clear_btn = gr.Button(
            "🗑️ Clear"
        )



    gr.Markdown("## 👨‍⚖️ Judge's Chamber")

    verdict_output = gr.Markdown()



    with gr.Tab("📊 Case Intelligence"):

        profile_output = gr.Markdown()

    with gr.Tab("⚖️ Strength Analysis"):

        strength_image = gr.Image(
            label="Case Strength",
            type="filepath"
        )

    with gr.Tab("📜 Case Timeline"):

        timeline_image = gr.Image(
            label="Case Timeline",
            type="filepath"
        )

    with gr.Tab("📄 Court Report"):

        report_output = gr.Textbox(
            label="Generated Court Report",
            lines=30,
            interactive=False
        )


    analyze_outputs = [
        verdict_output,
        profile_output,
        strength_image,
        timeline_image,
        report_output,
        gr.Textbox(visible=False)
    ]

    analyze_btn.click(
        fn=analyze_case,
        inputs=[
            case_input,
            prosecution_input,
            defense_input
        ],
        outputs=analyze_outputs
    )


    sample1_btn.click(
        fn=load_sample_1,
        outputs=[
            case_input,
            prosecution_input,
            defense_input
        ]
    )

    sample2_btn.click(
        fn=load_sample_2,
        outputs=[
            case_input,
            prosecution_input,
            defense_input
        ]
    )



    clear_btn.click(
        fn=lambda: (
            "",
            "",
            "",
            "",
            None,
            None,
            ""
        ),
        outputs=[
            case_input,
            prosecution_input,
            defense_input,
            verdict_output,
            strength_image,
            timeline_image,
            report_output
        ]
    )



app.launch(
    share=True,
    debug=False
)